In [ ]:
# Library Initialization
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import ImageGrid
import scipy as sp

import pyvista as pv
from pyvistaqt import BackgroundPlotter,MultiPlotter

# from matplotlib.widgets import Button, Slider

In [ ]:
# Helper Function

# Coords Helper
def C12(k,size):
    x=(k)%size[0]
    y=(k)//size[0]
    return x,y

def C13(k,j,size):
    i=k
    return i,j

def C21(pos,size):
    k=pos[0]+pos[1]*size[0]
    return k

def C23(pos,j,size):
    i=C21(pos,size)
    return i,j

def C31(i,j=0):
    return i

def C32(pos,size):
    x,y=C12(pos[0],size)
    return x,y

# Reconstruct Helper
def R12(par,size):
    i=np.array(range(size[0]*size[1]))
    val=np.zeros([size[1],size[0]],dtype=complex)
    x,y=C12(i,size)
    val[y,x]=par[i]
    return val

def R21(par,size):
    ipos=np.zeros([2,size[0]*size[1]],dtype=int)
    val=np.zeros(size[0]*size[1],dtype=complex)
    for j in range(size[1]):
        for i in range(size[0]):
            ipos[0,j*size[0]+i]=i
            ipos[1,j*size[0]+i]=j
    k=C21(ipos,size)
    r=range(size[0]*size[1])
    val[k]=par[ipos[0,r],ipos[1,r]]
    return val

# Hamiltonian Builder
def FDiffTheta(H,pos,a,dt,j,size):
    b=a/(2*dt)

    if pos[1]!=0:
        H[j,C21(pos+[0,-1],size)]-=b
    else:
        H[j,C21([pos[0],size[1]-1],size)]-=b
    if pos[1]!=(size[1]-1):
        H[j,C21(pos+[0, 1],size)]+=b
    else:
        H[j,C21([pos[0],0],size)]+=b
    return H

def SDiffTheta(H,pos,a,dt,j,size):
    b=a/dt**2

    H[j,C21(pos,size)]-=2*b
    if pos[1]!=0:
        H[j,C21(pos+[0,-1],size)]+=b
    else:
        H[j,C21([pos[0],size[1]-1],size)]+=b
    if pos[1]!=(size[1]-1):
        H[j,C21(pos+[0, 1],size)]+=b
    else:
        H[j,C21([pos[0],0],size)]+=b
    return H

def SDiffPhi(H,pos,a,dt,j,size):
    b=a/dt**2

    H[j,C21(pos,size)]-=2*b
    if pos[0]!=0:
        H[j,C21(pos+[-1,0],size)]+=b
    if pos[0]!=(size[0]-1):
        H[j,C21(pos+[ 1,0],size)]+=b
    return H


def maingeneralsph(phiE,ng,size,sphi=4*np.pi,stheta=np.pi,Isincludestate=False,GetHamiltonian=False,base=1,tpRatio=1,alpha=1,beta=1,Enum=4):
    N,M=size
    NM=N*M

    H=np.zeros([NM,NM],dtype=complex)
    phi=np.linspace(-sphi,sphi,N)
    theta=np.linspace(-stheta,stheta,M)
    dp=phi[1]-phi[0]
    dt=theta[1]-theta[0]

    GHz=1000000000

    Ecp=base*GHz
    Ect=tpRatio*base*GHz
    El=alpha*GHz
    Ej=beta*GHz

    count=0

    for j in range(M):
        for i in range(N):
            pos=np.array([i,j])
            H[count,C21(pos,size)]= +4*(Ect*ng**2)\
                                    +El*(phi[i]-np.pi*phiE)**2\
                                    -2*Ej*np.cos(phi[i])*np.cos(theta[j])
            H=SDiffPhi(H,pos,-4*Ecp,dp,count,size)
            H=FDiffTheta(H,pos,+1j*8*Ect*ng,dt,count,size)
            H=SDiffTheta(H,pos,-4*Ect,dt,count,size)
            count+=1
       
    if GetHamiltonian:
        return H
    if Isincludestate:
        a,b=sp.sparse.linalg.eigsh(H,Enum,which='SR')
        return a,b
    a,b=sp.sparse.linalg.eigsh(H,Enum,which='SR')
    return a
def maingeneralh(phiE,ng,size,sphi=4*np.pi,stheta=np.pi,Isincludestate=False,GetHamiltonian=False,base=1,tpRatio=1,alpha=1,beta=1):
    N,M=size
    NM=N*M

    H=np.zeros([NM,NM],dtype=complex)
    phi=np.linspace(-sphi,sphi,N)
    theta=np.linspace(-stheta,stheta,M)
    dp=phi[1]-phi[0]
    dt=theta[1]-theta[0]

    GHz=1000000000

    Ecp=base*GHz
    Ect=tpRatio*base*GHz
    El=alpha*GHz
    Ej=beta*GHz

    count=0
    for j in range(M):
        for i in range(N):
            pos=np.array([i,j])
            H[count,C21(pos,size)]= +4*(Ect*ng**2)\
                                    +El*(phi[i]-np.pi*phiE)**2\
                                    -2*Ej*np.cos(phi[i])*np.cos(theta[j])
            H=SDiffPhi(H,pos,-4*Ecp,dp,count,size)
            H=FDiffTheta(H,pos,+1j*8*Ect*ng,dt,count,size)
            H=SDiffTheta(H,pos,-4*Ect,dt,count,size)
            count+=1
    if GetHamiltonian:
        return H
    if Isincludestate:
        a,b=np.linalg.eigh(H)
        n=np.argsort(a.real)
        asort=a[n]
        wf=b[:,n]
        return asort,wf
    a=np.linalg.eigvalsh(H)
    n=np.argsort(a.real)
    asort=a[n]
    return asort

In [ ]:
# Single param varied Plotting
# Initialization part single variation 
# Note all param has better Plotting
# I have not update this part

En=np.load("Result/alpha-varied-result.npy")

# Set this depending on kind
# Key 1-Base, 2-tRatio, 3-Alpha, 4-Beta
key=1

# Comment used
param1=1
param2=1
# param3=1
param4=1


# Job Parameter
iphiE=0
ing=0
ibase=0
n1=7
n2=7
n3=15
nt=n1*n2
nE0=3

rilbase=np.linspace(0.2,5,n3)
tpRatio=np.linspace(1,5,n3)
alpha=np.linspace(0.2,10,n3)
beta=np.linspace(1/200,27.2,n3)
sSweeppE=0.5
sSweepng=0.5
phiE=np.linspace(-0,sSweeppE,n1)
ng=np.linspace(-0,sSweepng,n2)

# Uncomment used
# param1=rilbase[0]
# param2=tpRatio[0]
param3=alpha[0]
# param4=beta[0]


ix=np.arange(n1)
iy=np.arange(n2)

def IndexHelper(val,valmin,valmax,n):
    d=(valmax-valmin)/n
    return np.int32(np.round((val-valmin)/d))

phiE2i=lambda x: IndexHelper(x,phiE[0],phiE[n1-1],n1-1)
ng2i=lambda x: IndexHelper(x,ng[0],ng[n2-1],n2-1)
base2i=lambda x: IndexHelper(x,base[0],base[n3-1],n3-1)
tpRatio2i=lambda x: IndexHelper(x,tpRatio[0],tpRatio[n3-1],n3-1)
alpha2i=lambda x: IndexHelper(x,alpha[0],alpha[n3-1],n3-1)
beta2i=lambda x: IndexHelper(x,beta[0],beta[n3-1],n3-1)

def GetExtraIndex():
    return np.int64(round(ibase*nt))

def GetE(a,b):
    ii=np.add.outer(a,b).flatten()
    ii=np.int32(np.round(ii))
    Ee=En[ii].real
    return Ee,nE0

def coordsChanged(val,num,plot,scaMem):
    if num==0:
        global iphiE
        iphiE=phiE2i(val)
        a=iphiE
        b=iy*n1
        # plot[0,0].add_mesh(pv.Plane(center=(val,0.25,5E4),direction=(1,0,0)),name="phiE")
    else:
        global ing
        ing=ng2i(val)
        a=ix
        b=ing*n1
        # plot[0,0].add_mesh(pv.Plane(center=(0.25,val,5E4),direction=(0,1,0)),name="ng")
    Ee,n0=GetE(a+ibase*n1*n2,b)
    E0=np.full_like(Ee[:,1].real,0)
    E0=Ee[:,n0]
    scaMem[num,0].update(Ee[:,1-num],Ee[:,n0  ],)
    scaMem[num,1].update(Ee[:,1-num],Ee[:,n0  ]-E0,)
    scaMem[num,2].update(Ee[:,1-num],Ee[:,n0+1],)
    scaMem[num,3].update(Ee[:,1-num],Ee[:,n0+1]-E0,)
    scaMem[num,4].update(Ee[:,1-num],Ee[:,n0+2],)
    scaMem[num,5].update(Ee[:,1-num],Ee[:,n0+2]-E0,)
    scaMem[num,6].update(Ee[:,1-num],Ee[:,n0+3],)
    scaMem[num,7].update(Ee[:,1-num],Ee[:,n0+3]-E0,)

sargs = dict(height=0.25, vertical=True, position_x=0.0, position_y=0.5)
ap_args=dict(render_points_as_spheres=True,reset_camera=False,scalar_bar_args=sargs)
def baseChanged(val,plot=None):
    # single variation specific
    global ibase
    ibase=setParam[key-1](val)
    # Same as all varied
    Ee,n0=GetE(ix+ibase*nt,iy*n1)
    points0=np.transpose([Ee[:,0],Ee[:,1],Ee[:,n0  ]])
    points1=np.transpose([Ee[:,0],Ee[:,1],Ee[:,n0+1]])
    points2=np.transpose([Ee[:,0],Ee[:,1],Ee[:,n0+2]])
    points3=np.transpose([Ee[:,0],Ee[:,1],Ee[:,n0+3]])
    E0=np.zeros_like(points0)
    Emin=[0,0,Ee[:,n0].min()]
    E0[:,2]=Ee[:,n0]
    rng0=points3[:,2].max()-points0[:,2].min()
    rng1=(points3[:,2]-points0[:,2]).max()

    # plot[0,0].add_mesh(pv.Plane(center=(0.25,0.25,0.5),direction=(0,1,0)),name="phiE")
    # plot[0,0].add_mesh(pv.Plane(center=(0.25,0.25,5E3),direction=(1,0,0)),name="phiE")
    # plot[0,0].add_mesh(pv.Plane(center=(0.25,0.25,5E3),direction=(0,1,0)),name="ng")
    # Only look 1 quadrant
    plot[0,0].set_scale(1,1,1/rng1)
    plot[0,0].add_points(points=points0-Emin,scalars=[0]*n1*n2,name='E0' ,**ap_args)
    plot[0,0].add_points(points=points1-Emin,scalars=[1]*n1*n2,name='E1' ,**ap_args)
    plot[0,0].add_points(points=points2-Emin,scalars=[2]*n1*n2,name='E2' ,**ap_args)
    plot[0,0].add_points(points=points3-Emin,scalars=[3]*n1*n2,name='E3' ,**ap_args)
    plot[1,0].set_scale(1,1,1/rng1)
    plot[1,0].add_points(points=points0-E0  ,scalars=[0]*n1*n2,name='E00',**ap_args)
    plot[1,0].add_points(points=points1-E0  ,scalars=[1]*n1*n2,name='E10',**ap_args)
    plot[1,0].add_points(points=points2-E0  ,scalars=[2]*n1*n2,name='E20',**ap_args)
    plot[1,0].add_points(points=points3-E0  ,scalars=[3]*n1*n2,name='E30',**ap_args)
    plot[0,0].reset_camera(bounds=[0,0.5,0,.5,0,1])
    plot[1,0].reset_camera(bounds=[0,0.5,0,.5,0,1])
    coordsChanged(phiE[iphiE],0,plot,scaMem)
    coordsChanged(ng[ing],1,plot,scaMem)
    SetParam()

def init(plot):
    global scaMem
    Ee,n0=GetE(iphiE+ibase*n1*n2,ing*n1)
    E0=Ee[:,n0]
    temp=pv.Chart2D
    e={'x_label':'ng','y_label':'E'}
    d={'x_label':'phiE','y_label':'E'}
    chart=np.array([[temp(**e),temp(**e)],[temp(**d),temp(**d)]])
    E0n0=chart[0,0].scatter(Ee[:,0],Ee[:,n0  ],   )
    E000=chart[0,1].scatter(Ee[:,0],Ee[:,n0  ]-E0)
    E1n0=chart[0,0].scatter(Ee[:,0],Ee[:,n0+1],   )
    E100=chart[0,1].scatter(Ee[:,0],Ee[:,n0+1]-E0)
    E2n0=chart[0,0].scatter(Ee[:,0],Ee[:,n0+2]   )
    E200=chart[0,1].scatter(Ee[:,0],Ee[:,n0+2]-E0)
    E3n0=chart[0,0].scatter(Ee[:,0],Ee[:,n0+3]   )
    E300=chart[0,1].scatter(Ee[:,0],Ee[:,n0+3]-E0)
    Ee,n0=GetE(iphiE+ibase*nt,ing*n1)
    E0=Ee[:,n0]
    E0n1=chart[1,0].scatter(Ee[:,1],Ee[:,n0  ]   )
    E001=chart[1,1].scatter(Ee[:,1],Ee[:,n0  ]-E0)
    E1n1=chart[1,0].scatter(Ee[:,1],Ee[:,n0+1]   )
    E101=chart[1,1].scatter(Ee[:,1],Ee[:,n0+1]-E0)
    E2n1=chart[1,0].scatter(Ee[:,1],Ee[:,n0+2]   )
    E201=chart[1,1].scatter(Ee[:,1],Ee[:,n0+2]-E0)
    E3n1=chart[1,0].scatter(Ee[:,1],Ee[:,n0+3]   )
    E301=chart[1,1].scatter(Ee[:,1],Ee[:,n0+3]-E0)
    scaMem=np.array([[E0n0,E000,E1n0,E100,E2n0,E200,E3n0,E300]\
           ,[E0n1,E001,E1n1,E101,E2n1,E201,E3n1,E301]])
    plot[0,1].add_chart(chart[0,0],)
    plot[1,1].add_chart(chart[0,1],)
    plot[0,2].add_chart(chart[1,0],)
    plot[1,2].add_chart(chart[1,1],)

drawsize=[40,20]
def DrawWF(val=0):
    size=drawsize
    wffig=plt.figure()
    wfax=ImageGrid(wffig,212,nrows_ncols=(1,4),axes_pad=0.05,label_mode="1",share_all=True,cbar_location="right",\
        cbar_mode="single",)
    wf=maingeneralh(phiE[iphiE],ng[ing],size,stheta=np.pi,Isincludestate=True,base=param1,tpRatio=param2,alpha=param3,beta=param4)[1]
    im0=np.abs(R12(wf[:,0],size))
    im1=np.abs(R12(wf[:,1],size))
    im2=np.abs(R12(wf[:,2],size))
    im3=np.abs(R12(wf[:,3],size))
    bound=[1,4]
    #wffig,wfax=plt.subplots(1,4,figsize=[10,10],sharex=True,sharey=True,)
    #wffig.text(0.5,0.11,"$\\phi$($\\pi$)",ha="center",size=12)
    #wffig.text(0.08,0.5,"$\\theta$($\\pi$)",va="center",rotation='vertical',size=12)
    wfax[0].imshow(np.transpose(im0),extent=[-bound[0],bound[0],-bound[1],bound[1]],norm='linear')
    wfax[1].imshow(np.transpose(im1),extent=[-bound[0],bound[0],-bound[1],bound[1]],norm='linear')
    wfax[2].imshow(np.transpose(im2),extent=[-bound[0],bound[0],-bound[1],bound[1]],norm='linear')
    e=wfax[3].imshow(np.transpose(im3),extent=[-bound[0],bound[0],-bound[1],bound[1]],norm='linear')
    wfax[0].cax.colorbar(e)
    wfax[0].set_xlabel("$\\phi$($\\pi$)")
    wfax[0].set_ylabel("$\\theta$($\\pi$)")
    #wffig.colorbar(e)
    plotter=BackgroundPlotter()
    plotter.add_chart(pv.ChartMPL(wffig))
    # wffig.savefig("AAA.pdf",format="pdf")


setParam=[lambda x:base2i(x),lambda x:tpRatio2i(x),lambda x:alpha2i(x),lambda x:beta2i(x)]
def SetParam(val=0):
    global param1,param2,param3,param4,base
    match key:
        case 1:
            param1=rilbase[ibase]
            base=rilbase
        case 2:
            param2=tpRatio[ibase]
            base=tpRatio
        case 3:
            param3=alpha[ibase]
            base=alpha
        case _:
            param4=beta[ibase]
            base=beta
SetParam()



In [ ]:
# Run

iphiE=0
ing=0
ibase=0

boundsSpec={'bounds':[0,0.5,0,0.5,0,1],'axes_ranges':[0,0.5,0,0.5,0,1], 'location':'origin','grid':'back',\
    'xtitle':'phiE','ytitle':'ng','ztitle':'E','n_xlabels':4,'n_ylabels':4,'n_zlabels':4,'ticks':'inside',\
    'minor_ticks':False,'font_size':1}#"show_xlabels":False,"show_ylabels":False}
    #'show_xlabels':False,'show_ylabels':False,'show_zlabels':False,}
plottertest=MultiPlotter(nrows=2,ncols=3)
plottertest[0,0].show_axes()
plottertest[0,0].set_scale(1,1,1E-10)
plottertest[0,0].show_bounds(**boundsSpec)
plottertest[1,0].show_axes()
plottertest[1,0].set_scale(1,1,1E-10)
plottertest[1,0].show_bounds(**boundsSpec)
plottertest[1,0].add_slider_widget(
    lambda x:baseChanged(x,plottertest),
    rng=[base[0],base[n3-1]],
    value=base[0],
    title='Base Value',
    pointa=(0.025, 0.1),
    pointb=(0.31, 0.1),
    style='modern',
)
plottertest[0,0].add_checkbox_button_widget(
    lambda x: DrawWF(),
    position=(0.025, 0.1),
    color_on='blue',
    color_off='blue'
)
init(plottertest)

# plottertest[1,0].show_axes()
plottertest[0,1].set_scale(1,1E-10)
plottertest[0,2].set_scale(1,1E-10)
plottertest[1,1].set_scale(1,1E-10)
plottertest[1,2].set_scale(1,1E-10)
# plottertest[1,1].show_bounds(bounds=[0,0.5,0,1,0,0],xtitle='phiE',ytitle='ng')
plottertest[0,1].add_slider_widget(
    lambda x:coordsChanged(x,0,plottertest,scaMem),
    rng=[phiE[0],phiE[n1-1]],
    value=phiE[0],
    title='Phi Value',
    pointa=(0.025, 0.1),
    pointb=(0.31, 0.1),
    style='modern',
)
plottertest[0,2].add_slider_widget(
    lambda x:coordsChanged(x,1,plottertest,scaMem),
    rng=[ng[0],ng[n2-1]],
    value=ng[0],
    title='ng Value',
    pointa=(0.025, 0.1),
    pointb=(0.31, 0.1),
    style='modern',
)

In [ ]:

# wfdata=np.load("Result/another-Experiment-result-wf.npy")
lee=len(En)
lee,np.sqrt(lee)

In [ ]:
# All param varied Plotting
# Initialization part all varied

En=np.load("Result/all-varied-result.npy")


# Job Parameter
iphiE=0
ing=0
ibase=0
itpRatio=0
ialpha=0
ibeta=0

# n1=41
# n2=41
# n3=4
# nt=n1*n2
# nE0=2

n1=7
n2=7
n3=4
nt=n1*n2
nE0=6

base=np.linspace(0.2,5,n3)
tpRatio=np.linspace(1,5,n3)
alpha=np.linspace(0.2,10,n3)
beta=np.linspace(1/200,27.2,n3)
sSweeppE=0.5
sSweepng=0.5
phiE=np.linspace(-0,sSweeppE,n1)
ng=np.linspace(-0,sSweepng,n2)



ix=np.arange(n1)
iy=np.arange(n2)
sargs = dict(height=0.25, vertical=True, position_x=0.0, position_y=0.5)

def IndexHelper(val,valmin,valmax,n):
    d=(valmax-valmin)/n
    return np.int16(np.round((val-valmin)/d))

phiE2i=lambda x: IndexHelper(x,phiE[0],phiE[n1-1],n1-1)
ng2i=lambda x: IndexHelper(x,ng[0],ng[n2-1],n2-1)
base2i=lambda x: IndexHelper(x,base[0],base[n3-1],n3-1)
tpRatio2i=lambda x: IndexHelper(x,tpRatio[0],tpRatio[n3-1],n3-1)
alpha2i=lambda x: IndexHelper(x,alpha[0],alpha[n3-1],n3-1)
beta2i=lambda x: IndexHelper(x,beta[0],beta[n3-1],n3-1)

def GetExtraIndex():
    return np.int64(round(ibase*nt+itpRatio*nt*n3+ialpha*nt*n3**2+ibeta*nt*n3**3))

def GetE(a,b):
    ii=np.add.outer(a,b).flatten()
    ii=np.int32(np.round(ii))
    Ee=En[ii].real
    return Ee,nE0

def CoordsChanged(val,num,plot,scaMem):
    if num==0:
        global iphiE
        iphiE=phiE2i(val)
        a=iphiE
        b=iy*n1
        # plot[0,0].add_mesh(pv.Plane(center=(val,0.25,5E4),direction=(1,0,0)),name="phiE")
    else:
        global ing
        ing=ng2i(val)
        a=ix
        b=ing*n1
        # plot[0,0].add_mesh(pv.Plane(center=(0.25,val,5E4),direction=(0,1,0)),name="ng")
    Ee,n0=GetE(a+GetExtraIndex(),b)
    E0=np.zeros_like(Ee[:,1].real)
    E0=Ee[:,n0]
    scaMem[num,0].update(Ee[:,1-num],Ee[:,n0  ],)
    scaMem[num,1].update(Ee[:,1-num],Ee[:,n0  ]-E0,)
    scaMem[num,2].update(Ee[:,1-num],Ee[:,n0+1],)
    scaMem[num,3].update(Ee[:,1-num],Ee[:,n0+1]-E0,)
    scaMem[num,4].update(Ee[:,1-num],Ee[:,n0+2],)
    scaMem[num,5].update(Ee[:,1-num],Ee[:,n0+2]-E0,)
    scaMem[num,6].update(Ee[:,1-num],Ee[:,n0+3],)
    scaMem[num,7].update(Ee[:,1-num],Ee[:,n0+3]-E0,)

ap_args=dict(render_points_as_spheres=True,reset_camera=False,scalar_bar_args=sargs)
def ParameterChanged(plot):
    Ee,n0=GetE(ix+GetExtraIndex(),iy*n1)
    points0=np.transpose([Ee[:,0],Ee[:,1],Ee[:,n0  ]])
    points1=np.transpose([Ee[:,0],Ee[:,1],Ee[:,n0+1]])
    points2=np.transpose([Ee[:,0],Ee[:,1],Ee[:,n0+2]])
    points3=np.transpose([Ee[:,0],Ee[:,1],Ee[:,n0+3]])
    E0=np.zeros_like(points0)
    Emin=[0,0,Ee[:,n0].min()]
    E0[:,2]=Ee[:,n0]
    # rng0=points3[:,2].max()-points0[:,2].min()
    rng1=(points3[:,2]-points0[:,2]).max()

    # plot[0,0].add_mesh(pv.Plane(center=(0.25,0.25,0.5),direction=(0,1,0)),name="phiE")
    # plot[0,0].add_mesh(pv.Plane(center=(0.25,0.25,5E3),direction=(1,0,0)),name="phiE")
    # plot[0,0].add_mesh(pv.Plane(center=(0.25,0.25,5E3),direction=(0,1,0)),name="ng")
    plot.subplot(0,0)
    plot.set_scale(1,1,1/rng1)
    # Q1
    plot.add_points(points=points0-E0  ,scalars=[0]*n1*n2,name='E0Q1' ,**ap_args)
    plot.add_points(points=points1-E0  ,scalars=[1]*n1*n2,name='E1Q1' ,**ap_args)
    plot.add_points(points=points2-E0  ,scalars=[2]*n1*n2,name='E2Q1' ,**ap_args)
    plot.add_points(points=points3-E0  ,scalars=[3]*n1*n2,name='E3Q1' ,**ap_args)
    # Q2
    points0[:,0]=-points0[:,0]
    points1[:,0]=-points1[:,0]
    points2[:,0]=-points2[:,0]
    points3[:,0]=-points3[:,0]
    plot.add_points(points=points0-E0  ,scalars=[0]*n1*n2,name='E0Q2' ,**ap_args)
    plot.add_points(points=points1-E0  ,scalars=[1]*n1*n2,name='E1Q2' ,**ap_args)
    plot.add_points(points=points2-E0  ,scalars=[2]*n1*n2,name='E2Q2' ,**ap_args)
    plot.add_points(points=points3-E0  ,scalars=[3]*n1*n2,name='E3Q2' ,**ap_args)
    # Q3
    points0[:,1]=-points0[:,1]
    points1[:,1]=-points1[:,1]
    points2[:,1]=-points2[:,1]
    points3[:,1]=-points3[:,1]
    plot.add_points(points=points0-E0  ,scalars=[0]*n1*n2,name='E0Q3' ,**ap_args)
    plot.add_points(points=points1-E0  ,scalars=[1]*n1*n2,name='E1Q3' ,**ap_args)
    plot.add_points(points=points2-E0  ,scalars=[2]*n1*n2,name='E2Q3' ,**ap_args)
    plot.add_points(points=points3-E0  ,scalars=[3]*n1*n2,name='E3Q3' ,**ap_args)
    # Q4
    points0[:,0]=-points0[:,0]
    points1[:,0]=-points1[:,0]
    points2[:,0]=-points2[:,0]
    points3[:,0]=-points3[:,0]
    plot.add_points(points=points0-E0  ,scalars=[0]*n1*n2,name='E0Q4' ,**ap_args)
    plot.add_points(points=points1-E0  ,scalars=[1]*n1*n2,name='E1Q4' ,**ap_args)
    plot.add_points(points=points2-E0  ,scalars=[2]*n1*n2,name='E2Q4' ,**ap_args)
    plot.add_points(points=points3-E0  ,scalars=[3]*n1*n2,name='E3Q4' ,**ap_args)
    plot.reset_camera(bounds=[-0.5,0.5,-0.5,.5,0,1])
    plot.subplot(1,0)

    points0[:,1]=-points0[:,1]
    points1[:,1]=-points1[:,1]
    points2[:,1]=-points2[:,1]
    points3[:,1]=-points3[:,1]
    plot.add_points(points=points0-E0  ,scalars=[0]*n1*n2,name='E00',**ap_args)
    plot.add_points(points=points1-E0  ,scalars=[1]*n1*n2,name='E10',**ap_args)
    plot.add_points(points=points2-E0  ,scalars=[2]*n1*n2,name='E20',**ap_args)
    plot.add_points(points=points3-E0  ,scalars=[3]*n1*n2,name='E30',**ap_args)
    plot.reset_camera(bounds=[0,0.5,0,.5,0,1])
    CoordsChanged(phiE[iphiE],0,plot,scaMem)
    CoordsChanged(ng[ing],1,plot,scaMem)

def BaseChanged(val,plot):
    global ibase
    ibase=base2i(val)
    ParameterChanged(plot)

def tpRatioChanged(val,plot):
    global itpRatio
    itpRatio=tpRatio2i(val)
    ParameterChanged(plot)

def AlphaChanged(val,plot):
    global ialpha
    ialpha=alpha2i(val)
    ParameterChanged(plot)

def BetaChanged(val,plot):
    global ibeta
    ibeta=beta2i(val)
    ParameterChanged(plot)


def init(plot):
    global scaMem
    Ee,n0=GetE(iphiE+ibase*n1*n2,ing*n1)
    E0=Ee[:,n0]
    temp=pv.Chart2D
    e={'x_label':'ng','y_label':'E'}
    d={'x_label':'phiE','y_label':'E'}
    chart=np.array([[temp(**e),temp(**e)],[temp(**d),temp(**d)]])
    E0n0=chart[0,0].scatter(Ee[:,0],Ee[:,n0  ],   )
    E000=chart[0,1].scatter(Ee[:,0],Ee[:,n0  ]-E0)
    E1n0=chart[0,0].scatter(Ee[:,0],Ee[:,n0+1],   )
    E100=chart[0,1].scatter(Ee[:,0],Ee[:,n0+1]-E0)
    E2n0=chart[0,0].scatter(Ee[:,0],Ee[:,n0+2]   )
    E200=chart[0,1].scatter(Ee[:,0],Ee[:,n0+2]-E0)
    E3n0=chart[0,0].scatter(Ee[:,0],Ee[:,n0+3]   )
    E300=chart[0,1].scatter(Ee[:,0],Ee[:,n0+3]-E0)
    Ee,n0=GetE(iphiE+ibase*nt,ing*n1)
    E0=Ee[:,n0]
    E0n1=chart[1,0].scatter(Ee[:,1],Ee[:,n0  ]   )
    E001=chart[1,1].scatter(Ee[:,1],Ee[:,n0  ]-E0)
    E1n1=chart[1,0].scatter(Ee[:,1],Ee[:,n0+1]   )
    E101=chart[1,1].scatter(Ee[:,1],Ee[:,n0+1]-E0)
    E2n1=chart[1,0].scatter(Ee[:,1],Ee[:,n0+2]   )
    E201=chart[1,1].scatter(Ee[:,1],Ee[:,n0+2]-E0)
    E3n1=chart[1,0].scatter(Ee[:,1],Ee[:,n0+3]   )
    E301=chart[1,1].scatter(Ee[:,1],Ee[:,n0+3]-E0)
    scaMem=np.array([[E0n0,E000,E1n0,E100,E2n0,E200,E3n0,E300]\
           ,[E0n1,E001,E1n1,E101,E2n1,E201,E3n1,E301]])
    plot.subplot(0,0)
    plot.add_chart(chart[0,0],)
    plot.subplot(1,0)
    plot.add_chart(chart[0,1],)
    plot.subplot(0,1)
    plot.add_chart(chart[1,0],)
    plot.subplot(1,1)
    plot.add_chart(chart[1,1],)
    
drawsize=[40,20]
def DrawWF(val=0):
    if val==99:
        wf=wfdata
        size=drawsize
        im0=np.abs(R12(wf[:,0],size))
        im1=np.abs(R12(wf[:,1],size))
        im2=np.abs(R12(wf[:,2],size))
        im3=np.abs(R12(wf[:,3],size))
    else:
        size=drawsize
        wf=maingeneralh(phiE[iphiE],ng[ing],size,stheta=np.pi,Isincludestate=True,base=base[ibase],tpRatio=tpRatio[itpRatio],alpha=alpha[ialpha],beta=beta[ibeta])[1]
        im0=np.abs(R12(wf[:,0],size))
        im1=np.abs(R12(wf[:,1],size))
        im2=np.abs(R12(wf[:,2],size))
        im3=np.abs(R12(wf[:,3],size))
    wffig=plt.figure()
    wfax=ImageGrid(wffig,111,nrows_ncols=(1,4),axes_pad=0.05,label_mode="1",share_all=True,cbar_location="right",\
        cbar_mode="single",)
    bound=[1,4]
    #wffig,wfax=plt.subplots(1,4,figsize=[10,10],sharex=True,sharey=True,)
    #wffig.text(0.5,0.11,"$\\phi$($\\pi$)",ha="center",size=12)
    #wffig.text(0.08,0.5,"$\\theta$($\\pi$)",va="center",rotation='vertical',size=12)
    wfax[0].imshow(np.transpose(im0),extent=[-bound[0],bound[0],-bound[1],bound[1]],norm='linear')
    wfax[1].imshow(np.transpose(im1),extent=[-bound[0],bound[0],-bound[1],bound[1]],norm='linear')
    wfax[2].imshow(np.transpose(im2),extent=[-bound[0],bound[0],-bound[1],bound[1]],norm='linear')
    e=wfax[3].imshow(np.transpose(im3),extent=[-bound[0],bound[0],-bound[1],bound[1]],norm='linear')
    wfax[0].cax.colorbar(e)
    wfax[0].set_xlabel("$\\phi$($\\pi$)")
    wfax[0].set_ylabel("$\\theta$($\\pi$)")
    #wffig.colorbar(e)
    plotterwf.add_chart(pv.ChartMPL(wffig))





In [ ]:
# all varied plot initialization
plotter3d=BackgroundPlotter(shape=(2,1),show=True,window_size=(400,800),off_screen=False)
plotterchart=BackgroundPlotter(shape=(2,2),show=True,window_size=(800,800),off_screen=False)
init(plotterchart)
plotterwf=BackgroundPlotter(show=True,window_size=(400,400),off_screen=False)
Slider=BackgroundPlotter(window_size=(400,400),show=True,off_screen=False)
Slider.add_slider_widget(
    lambda x:BaseChanged(x,plotter3d),
    rng=[base[0],base[n3-1]],
    value=base[0],
    title='Base Value',
    pointa=(0.1, 0.9),
    pointb=(0.9, 0.9),
    style='modern',
)
Slider.add_slider_widget(
    lambda x:tpRatioChanged(x,plotter3d),
    rng=[tpRatio[0],tpRatio[n3-1]],
    value=tpRatio[0],
    title='tpRatio Value',
    pointa=(0.1, 0.75),
    pointb=(0.9, 0.75),
    style='modern',
)
Slider.add_slider_widget(
    lambda x:AlphaChanged(x,plotter3d),
    rng=[alpha[0],alpha[n3-1]],
    value=alpha[0],
    title='Alpha Value',
    pointa=(0.1, 0.6),
    pointb=(0.9, 0.6),
    style='modern',
)
Slider.add_slider_widget(
    lambda x:BetaChanged(x,plotter3d),
    rng=[beta[0],beta[n3-1]],
    value=beta[0],
    title='Beta Value',
    pointa=(0.1, 0.45),
    pointb=(0.9, 0.45),
    style='modern',
)

Slider.add_checkbox_button_widget(
    lambda x: DrawWF(),
    position=(0.1, 0.8),
    color_on='blue',
    color_off='blue',
    size=20
)
Slider.add_checkbox_button_widget(
    lambda x: DrawWF(99),
    position=(0.1, 300),
    color_on='blue',
    color_off='blue',
    size=20
)


Slider.add_slider_widget(
    lambda x:CoordsChanged(x,0,plotterchart,scaMem),
    rng=[phiE[0],phiE[n1-1]],
    value=phiE[0],
    title='Phi Value',
    pointa=(0.2, 0.3),
    pointb=(0.8, 0.3),
    style='modern',
)
Slider.add_slider_widget(
    lambda x:CoordsChanged(x,1,plotterchart,scaMem),
    rng=[ng[0],ng[n2-1]],
    value=ng[0],
    title='ng Value',
    pointa=(0.2, 0.15),
    pointb=(0.8, 0.15),
    style='modern',
)

boundsSpec={'bounds':[-0.5,0.5,-0.5,0.5,0,1],'axes_ranges':[-0.5,0.5,-0.5,0.5,0,1], 'location':'origin','grid':'back',\
    'xtitle':'phiE','ytitle':'ng','ztitle':'E','n_xlabels':4,'n_ylabels':4,'n_zlabels':4,'ticks':'inside',\
    'minor_ticks':False,'font_size':1}
    #'show_xlabels':False,'show_ylabels':False,'show_zlabels':False,}
plotter3d.subplot(0,0)
plotter3d.show_axes()
plotter3d.show_bounds(**boundsSpec)
plotter3d.subplot(1,0)
plotter3d.show_axes()
boundsSpec["bounds"]=[-0,0.5,-0,0.5,0,1]
boundsSpec["axes_ranges"]=[-0,0.5,-0,0.5,0,1]
plotter3d.show_bounds(**boundsSpec)

In [ ]:
# Reference value
sSweepng=0.5
sSweeppE=0.5

# For Ideal&Experimentally Realized
n1=21
n2=21
n3=15
phiE=np.linspace(-sSweeppE,sSweeppE,n1)
ng=np.linspace(-sSweepng,sSweepng,n2)


# For 1 param varied (default is 1)
n1=7
n2=7
n3=15
phiE=np.linspace(0,sSweeppE,n1)
ng=np.linspace(0,sSweepng,n2)
base=np.linspace(0.2,5,n3)
tpRatio=np.linspace(1,5,n3)
alpha=np.linspace(0.2,10,n3)
beta=np.linspace(1/200,27.2,n3)

# For all param varied
n1=7
n2=7
n3=4








In [ ]:
drawsize=[80,80]